# AB实验分析：新旧页面转化率对比

## 项目概述
对电商网站新页面设计进行A/B实验评估，数据集约29万条记录，目标为验证新页面是否显著提升用户转化率。

### 分析框架
```
数据校验 → 假设检验 → 辛普森悖论检查 → 决策耗时分析 → 功效分析 → 综合决策
```

### 数据来源
- 主实验数据：`ab_data748296.xlsx`（Kaggle Marketing A/B Test）
- 国家维度数据：`countries237256.xlsx`
- 约29万条记录，包含 user_id, timestamp, group, landing_page, converted 字段

## 第一步：数据加载与清洗

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 配置
DATA_PATH = './ab_data748296.xlsx'
COUNTRY_PATH = './countries237256.xlsx'
OUTPUT_DIR = './output/'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 加载数据
df = pd.read_excel(DATA_PATH)
countries = pd.read_excel(COUNTRY_PATH)
print(f'原始数据量: {len(df):,}')
print(f'国家数据量: {len(countries):,}')
print(f'列名: {df.columns.tolist()}')
df.head()

### 1.1 一致性校验与清洗

In [ ]:
# 一致性校验：control必须看old_page, treatment必须看new_page
mismatch_ctrl = ((df['group'] == 'control') & (df['landing_page'] == 'new_page')).sum()
mismatch_trt = ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page')).sum()
print(f'control组看到new_page: {mismatch_ctrl}')
print(f'treatment组看到old_page: {mismatch_trt}')
print(f'不一致记录总数: {mismatch_ctrl + mismatch_trt}')

# 剔除不一致记录
mask = ~(
    ((df['group'] == 'control') & (df['landing_page'] == 'new_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page'))
)
df = df[mask].copy()
print(f'剔除后剩余: {len(df):,}')

# 去重
dup_count = df['user_id'].duplicated(keep='first').sum()
print(f'重复user_id: {dup_count}')
df = df.drop_duplicates(subset='user_id', keep='first')
print(f'去重后剩余: {len(df):,}')

# 合并国家数据
df = df.merge(countries, on='user_id', how='inner')
print(f'合并国家数据后: {len(df):,}')

### 1.2 时间特征处理

In [ ]:
# timestamp仅含时间无日期，转为秒数
def time_to_seconds(t):
    if hasattr(t, 'hour'):
        return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
    return 0

df['time_seconds'] = df['timestamp'].apply(time_to_seconds)
df['time_min'] = df['time_seconds'] / 60
print('耗时统计（秒）：')
print(df['time_seconds'].describe())
print(f'\n⚠️ 注意：timestamp仅含时间无日期，无法做时间趋势分析')
print(f'✅ 清洗完成，最终可用数据: {len(df):,} 条')

## 第二步：描述性统计

In [ ]:
# 整体分组统计
grouped = df.groupby('group').agg(
    n=('converted', 'count'),
    conversions=('converted', 'sum'),
    conv_rate=('converted', 'mean')
)
grouped['conv_rate_pct'] = grouped['conv_rate'] * 100

print('整体分组统计：')
for g in ['control', 'treatment']:
    r = grouped.loc[g]
    print(f'  {g}: n={r["n"]:,}, 转化数={r["conversions"]:,}, 转化率={r["conv_rate_pct"]:.2f}%')

diff = grouped.loc['treatment','conv_rate_pct'] - grouped.loc['control','conv_rate_pct']
rel_diff = diff / grouped.loc['control','conv_rate_pct'] * 100
print(f'  差异: {diff:+.2f}pp (相对差异: {rel_diff:+.2f}%)')

In [ ]:
# 按国家分组统计
country_stats = df.groupby(['country', 'group']).agg(
    n=('converted', 'count'),
    conv_rate=('converted', 'mean')
).reset_index()
country_stats['conv_rate_pct'] = country_stats['conv_rate'] * 100

for country in ['US', 'UK', 'CA']:
    sub = country_stats[country_stats['country'] == country]
    ctrl = sub[sub['group'] == 'control'].iloc[0]
    treat = sub[sub['group'] == 'treatment'].iloc[0]
    diff_c = treat['conv_rate_pct'] - ctrl['conv_rate_pct']
    print(f'  {country}: 对照组 {ctrl["conv_rate_pct"]:.2f}% (n={ctrl["n"]:,}), '
          f'实验组 {treat["conv_rate_pct"]:.2f}% (n={treat["n"]:,}), 差异 {diff_c:+.2f}pp')

In [ ]:
# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

groups = ['Control\n(旧页面)', 'Treatment\n(新页面)']
rates = [grouped.loc['control', 'conv_rate_pct'], grouped.loc['treatment', 'conv_rate_pct']]
bars = axes[0].bar(groups, rates, color=['#4A90D9', '#E74C3C'], width=0.5)
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].set_title('Overall Conversion Rate')
axes[0].set_ylim(11.5, 12.5)
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{rate:.2f}%', ha='center', fontsize=12)

x = np.arange(3)
width = 0.3
ctrl_rates = [country_stats[(country_stats['country']==c) & (country_stats['group']=='control')]['conv_rate_pct'].values[0] for c in ['US','UK','CA']]
treat_rates = [country_stats[(country_stats['country']==c) & (country_stats['group']=='treatment')]['conv_rate_pct'].values[0] for c in ['US','UK','CA']]
axes[1].bar(x - width/2, ctrl_rates, width, label='Control', color='#4A90D9')
axes[1].bar(x + width/2, treat_rates, width, label='Treatment', color='#E74C3C')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['US', 'UK', 'CA'])
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].set_title('Conversion Rate by Country')
axes[1].legend()
axes[1].set_ylim(10.5, 13)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'descriptive_stats.png', dpi=150, bbox_inches='tight')
plt.show()

## 第三步：假设检验（双比例z检验）

In [ ]:
ctrl = df[df['group'] == 'control']
treat = df[df['group'] == 'treatment']

n1, n2 = len(ctrl), len(treat)
x1, x2 = ctrl['converted'].sum(), treat['converted'].sum()
p1, p2 = x1/n1, x2/n2

# 双比例z检验
p_pool = (x1 + x2) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z_stat = (p1 - p2) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

# 95%置信区间
se_diff = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
ci_lower = (p1 - p2) - 1.96 * se_diff
ci_upper = (p1 - p2) + 1.96 * se_diff

print(f'H₀: p_control = p_treatment')
print(f'H₁: p_control ≠ p_treatment (双尾)\n')
print(f'对照组转化率: {p1*100:.4f}%')
print(f'实验组转化率: {p2*100:.4f}%')
print(f'差异: {(p1-p2)*100:.4f}pp\n')
print(f'z统计量: {z_stat:.4f}')
print(f'p值: {p_value:.4f}')
print(f'95%置信区间: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]\n')

if p_value < 0.05:
    print(f'❌ 拒绝H₀，两组转化率有显著差异')
else:
    print(f'✅ 不拒绝H₀，两组转化率无显著差异')
    if ci_upper * 100 < 0.1:
        print(f'   注意：CI上界仅+{ci_upper*100:.2f}%，即使新页面有效果也极小')

In [ ]:
# 置信区间可视化
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(0, (p1-p2)*100, yerr=1.96*se_diff*100, fmt='o',
            color='#E74C3C', capsize=8, capthick=2, markersize=8)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlim(-0.5, 0.5)
ax.set_ylabel('Difference in Conversion Rate (pp)')
ax.set_title(f'95% CI: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]')
ax.text(0.15, (p1-p2)*100, f'p = {p_value:.4f}', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'hypothesis_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 第四步：辛普森悖论检查

In [ ]:
print('按国家分层z检验：\n')
simpson_results = []

for country in ['US', 'UK', 'CA']:
    sub = df[df['country'] == country]
    ctrl = sub[sub['group'] == 'control']
    treat = sub[sub['group'] == 'treatment']
    n1s, n2s = len(ctrl), len(treat)
    x1s, x2s = ctrl['converted'].sum(), treat['converted'].sum()
    p1s, p2s = x1s/n1s, x2s/n2s
    p_pool_s = (x1s + x2s) / (n1s + n2s)
    se_s = np.sqrt(p_pool_s * (1 - p_pool_s) * (1/n1s + 1/n2s))
    z_s = (p1s - p2s) / se_s
    p_val_s = 2 * (1 - stats.norm.cdf(abs(z_s)))
    direction = '对照组高' if p1s > p2s else '实验组高'
    sig = '✅ 显著' if p_val_s < 0.05 else '不显著'
    print(f'  {country}: 对照组 {p1s*100:.2f}% vs 实验组 {p2s*100:.2f}%, '
          f'差异 {(p1s-p2s)*100:+.2f}pp, 方向={direction}, z={z_s:.4f}, p={p_val_s:.4f} ({sig})')
    simpson_results.append({'country': country, 'ctrl_rate': p1s, 'treat_rate': p2s,
                           'diff_pp': (p1s-p2s)*100, 'direction': direction, 'z': z_s, 'p_value': p_val_s})

directions = set(r['direction'] for r in simpson_results)
if len(directions) > 1:
    print('\n⚠️ 存在辛普森悖论风险：各国家差异方向不一致！')
else:
    print('\n✅ 各层方向一致，无辛普森悖论风险')

In [ ]:
# 辛普森悖论可视化
fig, ax = plt.subplots(figsize=(8, 5))
countries_list = [r['country'] for r in simpson_results]
diffs = [r['diff_pp'] for r in simpson_results]
colors = ['#4A90D9' if d > 0 else '#E74C3C' for d in diffs]
bars = ax.bar(countries_list, diffs, color=colors, width=0.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
ax.set_ylabel('Difference (pp)')
ax.set_title('Conversion Rate Difference by Country\n(Positive = Control higher)')
for bar, d in zip(bars, diffs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{d:+.2f}pp', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'simpson_paradox.png', dpi=150, bbox_inches='tight')
plt.show()

## 第五步：用户决策耗时分析

In [ ]:
# 正态性检验
ctrl_time = df[df['group'] == 'control']['time_seconds']
treat_time = df[df['group'] == 'treatment']['time_seconds']

sample_c = ctrl_time.sample(5000, random_state=42)
sample_t = treat_time.sample(5000, random_state=42)
ks_c, p_ks_c = stats.kstest(sample_c, 'norm', args=(sample_c.mean(), sample_c.std()))
ks_t, p_ks_t = stats.kstest(sample_t, 'norm', args=(sample_t.mean(), sample_t.std()))
print(f'对照组KS检验: KS={ks_c:.4f}, p={p_ks_c:.4f}')
print(f'实验组KS检验: KS={ks_t:.4f}, p={p_ks_t:.4f}')
print(f'结论: 耗时不服从正态分布，应使用Mann-Whitney U非参数检验')

In [ ]:
# 全体用户 & 转化用户耗时比较
u_stat, u_pval = stats.mannwhitneyu(ctrl_time, treat_time, alternative='two-sided')
print(f'全体用户耗时: Mann-Whitney U p={u_pval:.4f}')

df_conv = df[df['converted'] == 1]
c_conv = df_conv[df_conv['group'] == 'control']['time_seconds']
t_conv = df_conv[df_conv['group'] == 'treatment']['time_seconds']
u_stat2, u_pval2 = stats.mannwhitneyu(c_conv, t_conv, alternative='two-sided')
print(f'转化用户耗时: Mann-Whitney U p={u_pval2:.4f}')
print(f'结论: 新页面未显著缩短决策时间')

In [ ]:
# 逻辑回归：耗时+分组→转化率
X_lr = pd.DataFrame({'const': 1, 'time_seconds': df['time_seconds'],
                      'group_num': (df['group'] == 'treatment').astype(int)})
y_lr = df['converted']
logit1 = sm.Logit(y_lr, X_lr).fit(disp=0)
print('逻辑回归（无交互项）：')
print(logit1.summary().tables[1])

# 含交互项
X_lr2 = pd.DataFrame({'const': 1, 'time_seconds': df['time_seconds'],
                       'group_num': (df['group'] == 'treatment').astype(int),
                       'interaction': df['time_seconds'] * (df['group'] == 'treatment').astype(int)})
logit2 = sm.Logit(y_lr, X_lr2).fit(disp=0)
print('\n逻辑回归（含交互项 time×group）：')
print(logit2.summary().tables[1])

In [ ]:
# 耗时分析可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(ctrl_time/60, bins=60, alpha=0.5, label='Control', color='#4A90D9', density=True)
axes[0].hist(treat_time/60, bins=60, alpha=0.5, label='Treatment', color='#E74C3C', density=True)
axes[0].set_xlabel('Time (min)')
axes[0].set_ylabel('Density')
axes[0].set_title('Decision Time Distribution')
axes[0].legend()

buckets = [(0,10),(10,20),(20,30),(30,40),(40,50),(50,60)]
ctrl_rates = [df[(df['time_min']>=lo)&(df['time_min']<hi)&(df['group']=='control')]['converted'].mean()*100 for lo,hi in buckets]
treat_rates = [df[(df['time_min']>=lo)&(df['time_min']<hi)&(df['group']=='treatment')]['converted'].mean()*100 for lo,hi in buckets]
x = np.arange(len(buckets))
width = 0.3
axes[1].bar(x - width/2, ctrl_rates, width, label='Control', color='#4A90D9')
axes[1].bar(x + width/2, treat_rates, width, label='Treatment', color='#E74C3C')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{lo}-{hi}' for lo,hi in buckets], rotation=30)
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].set_title('Conversion Rate by Time Bucket')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 第六步：功效分析

In [ ]:
p1_baseline = df[df['group'] == 'control']['converted'].mean()
current_n = len(df[df['group'] == 'control'])

print(f'基线转化率: {p1_baseline*100:.4f}%')
print(f'每组样本量: {current_n:,}\n')

print('不同MDE下每组所需样本量：')
print(f'{"MDE(相对)":>10} | {"MDE(绝对pp)":>12} | {"所需样本量/组":>14} | {"当前/所需":>8}')
print('-' * 55)

power = NormalIndPower()
for mde_rel in [0.01, 0.02, 0.05, 0.10]:
    p2 = p1_baseline * (1 + mde_rel)
    effect_size = proportion_effectsize(p1_baseline, p2)
    n_required = power.solve_power(effect_size=effect_size, alpha=0.05, power=0.80, ratio=1.0)
    ratio = current_n / n_required
    print(f'{mde_rel*100:>9.0f}% | {(p2-p1_baseline)*100:>11.4f}% | {n_required:>14,.0f} | {ratio:>8.2f}')

In [ ]:
# 功效曲线
p2_target = p1_baseline * 1.01
effect_size = proportion_effectsize(p1_baseline, p2_target)
n_required = power.solve_power(effect_size=effect_size, alpha=0.05, power=0.80, ratio=1.0)

fig, ax = plt.subplots(figsize=(8, 5))
sample_sizes = np.linspace(10000, 2000000, 100)
power_vals = [power.solve_power(effect_size=effect_size, alpha=0.05, nobs1=n, ratio=1.0) for n in sample_sizes]
ax.plot(sample_sizes/1e6, power_vals, color='#4A90D9', linewidth=2)
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='Power = 0.80')
ax.axvline(x=current_n/1e6, color='orange', linestyle=':', alpha=0.7, label=f'Current n={current_n:,}')
ax.axvline(x=n_required/1e6, color='green', linestyle=':', alpha=0.7, label=f'Required n={n_required:,.0f}')
ax.set_xlabel('Sample Size per Group (million)')
ax.set_ylabel('Statistical Power')
ax.set_title('Power Curve (MDE=1% Relative Lift)')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'power_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 第七步：综合决策

In [ ]:
print('=' * 60)
print('综合决策评估')
print('=' * 60)
print()
print('📋 最终决策：不上线新页面，停止当前实验\n')
print('理由：')
print('  1. 转化率差异不显著，95%CI上界仅+0.08%，无业务价值')
print('  2. 决策耗时两组无显著差异，新页面未提升效率')
print('  3. 存在辛普森悖论风险，不宜全局推广')
print('  4. 样本量差8倍，功效严重不足')
print('  5. timestamp缺日期，无法排除周期效应')
print()
print('后续建议：')
print('  - 重新设计实验：明确MDE、补全日期字段、规划合理周期')
print('  - 可对UK市场单独做小规模实验验证反转信号')
print('  - 优先评估新页面的定性价值（用户体验、品牌形象）')